# AEM4L2 E08 - Pipeline completo: de cero a confiable

**Objetivo:** juntar las bases: audio, ASR, errores, WER, error crítico y gate.

Este notebook es el cierre: no intenta ser difícil, intenta mostrar cómo se conectan todas las piezas.

**Python exercise relacionado:** `python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/e06_audio_pipeline_confiable.py`


## Resumen de piezas

| Pieza | Pregunta |
|---|---|
| Audio | ¿se escucha bien? |
| ASR | ¿qué texto salió? |
| Reference | ¿qué debería haber salido? |
| WER | ¿cuánto se equivocó? |
| Error crítico | ¿se equivocó en algo peligroso? |
| Gate | ¿automatizamos o revisamos? |


In [ ]:
golden_cases = [
    {
        "id": "support_clean",
        "domain": "ecommerce",
        "reference": "el pedido cuatro cinco dos uno no llego",
        "hypothesis": "el pedido cuatro cinco dos uno no llego",
    },
    {
        "id": "medical_critical",
        "domain": "medical",
        "reference": "tomar medicacion cada ocho horas",
        "hypothesis": "tomar medicacion cada dos horas",
    },
    {
        "id": "financial_amount",
        "domain": "finance",
        "reference": "transferir mil pesos hoy",
        "hypothesis": "transferir millon pesos hoy",
    },
]

for case in golden_cases:
    print(case['id'], '->', case['domain'])


## Golden cases

Un **golden case** es un caso de referencia que usamos una y otra vez para comparar versiones del sistema.

Buen golden set:

| Criterio | Qué significa |
|---|---|
| Representativo | aparece en uso real |
| Diverso | incluye ruido, acentos, dominios |
| Revisado | transcript humano confiable |
| Estable | no cambia cada semana |

Sin golden cases, no sabemos si una mejora realmente mejoró.


In [ ]:
def simple_wer(reference: str, hypothesis: str) -> float:
    ref = reference.split()
    hyp = hypothesis.split()
    # Versión didáctica: cuenta diferencias posicionales + diferencia de longitud.
    substitutions = sum(1 for a, b in zip(ref, hyp) if a != b)
    length_diff = abs(len(ref) - len(hyp))
    return (substitutions + length_diff) / len(ref) if ref else 0.0

for case in golden_cases:
    case['wer'] = simple_wer(case['reference'], case['hypothesis'])
    print(case['id'], f"WER={case['wer']:.2%}")


## Error crítico

WER puede ser bajo y aun así el caso puede ser peligroso.

Regla didáctica:

| Dominio | Términos críticos |
|---|---|
| medical | ocho, dos, dosis, no, sin, miligramos |
| finance | mil, millón, comprar, vender, transferir |
| legal | acepta, rechaza, plazo, rescindir |

En producción esto se vuelve una política de riesgo más seria.


In [ ]:
CRITICAL_TERMS = {
    'medical': {'ocho', 'dos', 'dosis', 'no', 'sin', 'miligramos'},
    'finance': {'mil', 'millon', 'millón', 'comprar', 'vender', 'transferir'},
    'ecommerce': {'cancelar', 'cambiar', 'urgente'},
}

def has_critical_change(case: dict) -> bool:
    terms = CRITICAL_TERMS.get(case['domain'], set())
    ref_terms = set(case['reference'].split())
    hyp_terms = set(case['hypothesis'].split())
    changed_terms = (ref_terms ^ hyp_terms) & terms
    return bool(changed_terms)

for case in golden_cases:
    case['critical_error'] = has_critical_change(case)
    print(case['id'], 'critical_error=', case['critical_error'])


## Gate de confiabilidad / reliability gate

Un gate combina métrica + política.

```text
si WER alto -> revisión humana
si error crítico -> revisión humana
si WER bajo y sin error crítico -> automatizar
```

La decisión no depende solo del LLM.


In [ ]:
THRESHOLDS = {
    'ecommerce': 0.20,
    'medical': 0.05,
    'finance': 0.05,
}

for case in golden_cases:
    threshold = THRESHOLDS[case['domain']]
    accepted = case['wer'] <= threshold and not case['critical_error']
    case['decision'] = 'automatizar' if accepted else 'revision_humana'
    print(f"{case['id']:<18} WER={case['wer']:.2%} threshold={threshold:.0%} critical={case['critical_error']} -> {case['decision']}")


## Gráfico simple de WER

La visualización ayuda a que el aula vea rápido qué casos se pasan de umbral.


In [ ]:
for case in golden_cases:
    wer = case['wer']
    threshold = THRESHOLDS[case['domain']]
    bar = '█' * round(wer * 30)
    mark = 'OK' if case['decision'] == 'automatizar' else 'REVISAR'
    print(f"{case['id']:<18} {bar:<30} WER={wer:.2%} umbral={threshold:.0%} {mark}")


## Qué guardar en producción

| Campo | Por qué importa |
|---|---|
| audio_file | rastrear evidencia |
| asr_model | comparar modelos |
| transcript | auditar salida ASR |
| reference | calcular WER si existe |
| WER | medir calidad |
| critical_error | riesgo de dominio |
| decision | explicar automatización o revisión |
| timestamp | monitoreo temporal |

Frase docente: **si no lo guardás, no lo podés auditar.**


## Cierre de L2

La progresión completa es:

```text
1. Entiendo las etapas.
2. Comparo referencia vs hypothesis.
3. Entiendo tokenización.
4. Clasifico errores ASR.
5. Calculo WER.
6. Estructuro el transcript.
7. Entiendo modelos posibles.
8. Decido con un gate.
```

Pregunta final:

> ¿Cuándo es mejor que el sistema no responda automáticamente?
